In [0]:
dbutils.widgets.removeAll()

In [0]:
# Debug values - replace with actual values when testing
import json

payload = {
    "ClientID": "01",
    "FileID": "01",
    "FileName": "/Workspace/Repos/logi@openhealthagents.org/alphaesai/ClaimsProcessing/temp/834/mickey.csv",
    "ClientContainer": "/Workspace/Repos/logi@openhealthagents.org/alphaesai/ClaimsProcessing/temp/834/",
    "CurrentFolderPath": "/Workspace/Repos/logi@openhealthagents.org/alphaesai/ClaimsProcessing/temp/834/current/",
    "ProcessedFolderPath": "/Volumes/claimsprocessing/bronze/member",
    "ColumnDelimiter": ",",
    "HasHeader": "true",
    "IgnoreHeader": "False",
    "FileLayoutID": "834",
    "FileLayoutDescription": "Standard834",
    "SchemaFileName": "MemberSchema.json",
    "SchemaFilePath": "/Workspace/Repos/logi@openhealthagents.org/alphaesai/ClaimsProcessing/DimMember/Bronze/Schema/MemberSchema.json",
    "TextQualifier": "\""
}
FileIds = json.dumps({"FileIds": payload})

In [0]:
from pyspark.sql import DataFrame

# lit used to add a constant value in a df
from pyspark.sql.functions import explode, col, lit
 
# StrucType - defines overall layout of the table
# StructField - defines a column
# StringType - defines a specific data inside that column
from pyspark.sql.types import StructType, StructField, StringType

In [0]:
def isIgnoreHeader(file: str, validation: str, delimiter: str, textQualifier: str):

    # 1. Load the JSON validation schema configuration
    # fullschema: DataFrame[Rules: array<struct<Data_file_query:string,Name:string,control_file_query:string>>, columnNames: array<struct<DataType:string,FieldName:string,Format:string,OrdinalPosition:bigint,Validators:array<struct<ConditionCol:string,ConditionVal:array<string>,InputValue:array<string>,Name:string,Value:string>>>>] 
    fullSchema = (spark.read.format("json").option("multiline", "true").load(validation))

    # 2. Extract and clean up the FieldNames
    # parsedSchema: DataFrame[FieldName: string, DataType: string]
    parsedSchema = (fullSchema.select(explode(col("columnNames"))).select(col("col.FieldName"), col("col.DataType")).filter(col("col.FieldName") != "TEMPLATE"))

    # 3. Collect the field names into a Python list
    # header: ['UNIQUEPERSONKEY', 'BENEFICIARYID', 'MEDICAIDID', 'LASTNAME', 'FIRSTNAME', 'MIDDLEINITIAL', 'DATEOFBIRTH', 'GENDER', 'PERMANENTADDRESSLINE1', 'PERMANENTADDRESSLINE2', 'PERMANENTCITY',...,'ALTERNATEKEY8', 'ALTERNATEKEY9', 'ALTERNATEKEY10', 'PLANMEMBERID', 'SUBSCRIBERID', 'STARTDATE', 'ENDDATE', 'SUBSCRIBERFLAG', 'FULLNAME', 'TITLE', 'SUFFIX', ...] 
    header = [row[0].strip() for row in parsedSchema.select("FieldName").collect()]

    # 4. Build the dynamic Spark schema structure
    # fields: [StructField('UNIQUEPERSONKEY', StringType(), True), StructField('BENEFICIARYID', StringType(), True), StructField('MEDICAIDID', StringType(), True), StructField('LASTNAME', StringType(), True),..., StructField('FULLNAME', StringType(), True), StructField('TITLE', StringType(), True), StructField('SUFFIX', StringType(), True), ...] 
    fields = [StructField(fieldName, StringType(), nullable=True) for fieldName in header]

    # schema: StructType([StructField('UNIQUEPERSONKEY', StringType(), True), StructField('BENEFICIARYID', StringType(), True), StructField('MEDICAIDID', StringType(), True), StructField('LASTNAME', StringType(), True), ..., StructField('THIRDPARTYLIABILITYTRNDATE', StringType(), True), StructField('OWEDPREMIUM', StringType(), True), StructField('PRODUCTID', StringType(), True)]) 
    schema = StructType(fields)

    # 5. Read the target CSV file using the built schema
    # dfFile1: DataFrame[UNIQUEPERSONKEY: string, BENEFICIARYID: string, MEDICAIDID: string, LASTNAME: string, FIRSTNAME: string, MIDDLEINITIAL: string, DATEOFBIRTH: string, GENDER: string, PERMANENTADDRESSLINE1: string,..., IKAUNIQUEID: string, FAMILYLINKINDICATOR: string, INPREMIUMDELINQUENCYPROCESS: string, THIRDPARTYLIABILITYEFFDATE: string, THIRDPARTYLIABILITYTRNDATE: string, OWEDPREMIUM: string, PRODUCTID: string] 
    dfFile1 = (
        spark.read.format("csv")
        .schema(schema)
        .option("header", False)
        .option("delimiter", delimiter)
        .option("quote", textQualifier)
        .load(file)
    )

    # 6. Drop the header row dynamically using the first column name
    first_col_name = header[0]  # Gets the dynamic string name of the 1st column
    dfFile = dfFile1.filter(col(first_col_name) != first_col_name)

    # dfFile: DataFrame[UNIQUEPERSONKEY: string, BENEFICIARYID: string, MEDICAIDID: string, LASTNAME: string, FIRSTNAME: string, MIDDLEINITIAL: string, DATEOFBIRTH: string, GENDER: string, PERMANENTADDRESSLINE1: string, PERMANENTADDRESSLINE2: string, PERMANENTCITY: string, PERMANENTSTATE: string, PERMANENTZIPCODE: string, ..., INPREMIUMDELINQUENCYPROCESS: string, THIRDPARTYLIABILITYEFFDATE: string, THIRDPARTYLIABILITYTRNDATE: string, OWEDPREMIUM: string, PRODUCTID: string] 
    return dfFile

In [0]:
# Call isIgnoreHeader instead (works with Spark Connect)
df_result = isIgnoreHeader(
    file=f"file:{payload['FileName']}",          
    validation=f"file:{payload['SchemaFilePath']}", 
    delimiter=payload["ColumnDelimiter"],
    textQualifier=payload["TextQualifier"]
)

display(df_result)

In [0]:
# use when csv don't have header
def withoutHeader(file: str, validation: str, delimiter: str, textQualifier: str) -> DataFrame:

    # fullSchema: DataFrame[Rules: array<struct<Data_file_query:string,Name:string,control_file_query:string>>, columnNames: array<struct<DataType:string,FieldName:string,Format:string,OrdinalPosition:bigint,Validators:array<struct<ConditionCol:string,ConditionVal:array<string>,InputValue:array<string>,Name:string,Value:string>>>>] 
    fullSchema = spark.read.format("json").option("multiline", "true").load(validation)
    
    # parsedSchema: DataFrame[FieldName: string, DataType: string] 
    parsedSchema = fullSchema.select(explode(col("columnNames"))).select(col("col.FieldName"), col("col.DataType")).filter(col("col.FieldName") != "TEMPLATE")

    # header: ['UNIQUEPERSONKEY', 'BENEFICIARYID', 'MEDICAIDID', 'LASTNAME', 'FIRSTNAME', 'MIDDLEINITIAL', 'DATEOFBIRTH', 'GENDER', 'PERMANENTADDRESSLINE1', 'PERMANENTADDRESSLINE2', 'PERMANENTCITY', 'PERMANENTSTATE', 'PERMANENTZIPCODE', 'PERMANENTCOUNTY', ..., ALTERNATEKEY8', 'ALTERNATEKEY9', 'ALTERNATEKEY10', 'PLANMEMBERID', 'SUBSCRIBERID', 'STARTDATE', 'ENDDATE', 'SUBSCRIBERFLAG', 'FULLNAME', 'TITLE', 'SUFFIX', ...]
    header = [row[0].strip() for row in parsedSchema.select("FieldName").collect()]

    # fields: [StructField('UNIQUEPERSONKEY', StringType(), True), StructField('BENEFICIARYID', StringType(), True), StructField('MEDICAIDID', StringType(), True), StructField('LASTNAME', StringType(), True), StructField('FIRSTNAME', StringType(), True), ...., StructField('STARTDATE', StringType(), True), StructField('ENDDATE', StringType(), True), StructField('SUBSCRIBERFLAG', StringType(), True), StructField('FULLNAME', StringType(), True), StructField('TITLE', StringType(), True), StructField('SUFFIX', StringType(), True), ...] 
    fields = [StructField(fieldName, StringType(), nullable=True) for fieldName in header]

    # schema: StructType([StructField('UNIQUEPERSONKEY', StringType(), True), StructField('BENEFICIARYID', StringType(), True), StructField('MEDICAIDID', StringType(), True), StructField('LASTNAME', StringType(), True), StructField('FIRSTNAME', StringType(), True), StructField('MIDDLEINITIAL', StringType(), True), ..., StructField('OWEDPREMIUM', StringType(), True), StructField('PRODUCTID', StringType(), True)]) 
    schema = StructType(fields)

    # dfFile: DataFrame[UNIQUEPERSONKEY: string, BENEFICIARYID: string, MEDICAIDID: string, LASTNAME: string, FIRSTNAME: string, MIDDLEINITIAL: string, DATEOFBIRTH: string, GENDER: string, PERMANENTADDRESSLINE1: string, PERMANENTADDRESSLINE2: string, PERMANENTCITY: string, PERMANENTSTATE: string, PERMANENTZIPCODE: string, PERMANENTCOUNTY: string, ..., FAMILYLINKINDICATOR: string, INPREMIUMDELINQUENCYPROCESS: string, THIRDPARTYLIABILITYEFFDATE: string, THIRDPARTYLIABILITYTRNDATE: string, OWEDPREMIUM: string, PRODUCTID: string] 
    dfFile = spark.read.format("csv").schema(schema).option("header", False).option("delimiter", delimiter).option("quote", textQualifier).load(file)

    return dfFile

In [0]:
def delimitedFile(file: str, validation: str, header: str, delimiter: str, textQualifier: str) -> DataFrame:

    # Read csv with all columns first (let spark infer structure)
    # DataFrame[UNIQUEPERSONKEY: string, BENEFICIARYID: string, MEDICAIDID: string, PLANMEMBERID: string, SUBSCRIBERID: string, ENROLLEEUNIQUEID: string, MASKEDMEMBERID: string, ALTERNATEKEY3: string, ALTERNATEKEY4: string, ALTERNATEKEY5: string, ..., CARETAKERLASTNAME: string, CARETAKERMIDDLEINITIAL: string]
    df_raw = spark.read.format("csv") \
        .option("header", header) \
        .option("delimiter", delimiter) \
        .option("quote", textQualifier) \
        .load(file)

    # drop TEMPLATE column if it exists
    if "TEMPLATE" in df_raw.columns:
        df_raw = df_raw.drop("TEMPLATE")

    # load schema definition (excluding TEMPLATE)
    # DataFrame[Rules: array<struct<Data_file_query:string,Name:string,control_file_query:string>>, columnNames: array<struct<DataType:string,FieldName:string,Format:string,OrdinalPosition:bigint,Validators:array<struct<ConditionCol:string,ConditionVal:array<string>,InputValue:array<string>,Name:string,Value:string>>>>]
    fullSchema = spark.read.format("json").option("multiline", "true").load(validation)

    #  parsedSchema: DataFrame[FieldName: string, DataType: string] 
    parsedSchema = fullSchema.select(explode(col("columnNames"))).select(col("col.FieldName"), col("col.DataType")).filter(col("col.FieldName") != "TEMPLATE")

    # schemaHeader: ['UNIQUEPERSONKEY', 'BENEFICIARYID', 'MEDICAIDID', 'LASTNAME', 'FIRSTNAME', 'MIDDLEINITIAL', 'DATEOFBIRTH', 'GENDER', 'PERMANENTADDRESSLINE1', 'PERMANENTADDRESSLINE2', 'PERMANENTCITY', 'PERMANENTSTATE', 'PERMANENTZIPCODE', ..., 'ALTERNATEKEY6', 'ALTERNATEKEY7', 'ALTERNATEKEY8', 'ALTERNATEKEY9', 'ALTERNATEKEY10', 'PLANMEMBERID', 'SUBSCRIBERID', 'STARTDATE', 'ENDDATE', 'SUBSCRIBERFLAG', 'FULLNAME', 'TITLE', 'SUFFIX', ...] 
    schemaHeader = [row[0].strip() for row in parsedSchema.select("FieldName").collect()]

    # select columns in schema order (add missing columns as NULL)
    # select_exprs (after one loop): [Column<'CAST(UNIQUEPERSONKEY AS STRING)'>] 
    select_exprs = []
    for col_name in schemaHeader:
        if col_name in df_raw.columns:
            select_exprs.append(col(col_name).cast(StringType()))
        else:
            select_exprs.append(lit(None).cast(StringType()).alias(col_name))

    # dfFile: DataFrame[UNIQUEPERSONKEY: string, BENEFICIARYID: string, MEDICAIDID: string, LASTNAME: string, FIRSTNAME: string, MIDDLEINITIAL: string, DATEOFBIRTH: string, GENDER: string, PERMANENTADDRESSLINE1: string, PERMANENTADDRESSLINE2: string, PERMANENTCITY: string, PERMANENTSTATE: string, PERMANENTZIPCODE: string, ...
    dfFile = df_raw.select(select_exprs)

    return dfFile

In [0]:
def path_exists(pathToCheck: str) -> bool:

    # References the JVM gateway on the cluster via PySpark sparkContext to securely invoke the Hadoop FileSystem APIs
    sc = spark.sparkContext
    path_class = sc._gateway.jvm.org.apache.hadoop.fs.Path
    fs_class = sc._gateway.jvm.org.apache.hadoop.fs.FileSystem

    fs = fs_class.get(sc._jsc.hadoopConfiguration())
    IsExists = fs.exists(path_class(pathToCheck))
    
    return IsExists

In [0]:
# Call delimitedFile instead (works with Spark Connect)
df_result = delimitedFile(
    file=f"file:{payload['FileName']}",          
    validation=f"file:{payload['SchemaFilePath']}", 
    header="True",
    delimiter=payload["ColumnDelimiter"],
    textQualifier=payload["TextQualifier"]
)

display(df_result)

In [0]:
# Call withoutHeader instead (works with Spark Connect)
df_result = withoutHeader(
    file=f"file:{payload['FileName']}",          
    validation=f"file:{payload['SchemaFilePath']}", 
    delimiter=payload["ColumnDelimiter"],
    textQualifier=payload["TextQualifier"]
)

display(df_result)